# 08 — Mark 4E — Checkpoint Fusion Validation

[![Phase](https://img.shields.io/badge/Pipeline-Mark%201%20to%204E-blue.svg)]()
[![Mode](https://img.shields.io/badge/Default-REUSE%20(fast%2C%20deterministic)-success.svg)]()

**Pipeline position:** notebook **08 of 09** — run the suite in order 00 → 09.
**Original:** `mark 1/mark_4e_checkpoint_fusion_validation.ipynb`

## Objective

Can fusing the two Mark 4C arms (control + recall-loss checkpoints) satisfy every final target simultaneously? Sweep the fusion weight and the global threshold, then select under the preregistered policy (precision-first, V116-first, balanced).

## Inputs (read-only)

- `mark_4d_gate_result.json`, Mark 4C control + recall-loss checkpoints & caches
- Frozen fusion inputs from `mark 1/mark_4c_outputs/` + `mark_1/mark_4d_outputs/` (REUSE)

## Outputs → `Evaluation/mark_1_to_4e_outputs/mark_4e_outputs/`

Every file below keeps the exact naming used by the archived run, so results are
directly comparable with the original `mark 1/mark_*_outputs/` outputs.

| File |
|---|
| `mark_4e_gate_result.json` |
| `fusion_threshold_results.csv` |
| `fusion_patient_metrics.csv` |
| `fusion_positive_slice_diagnostic.csv` |
| `best_configuration_by_policy.csv` |
| `selected_gate_table.csv` |
| `fusion_validation_dashboard.png` |
| `selected_fusion_patient_heatmap.png` |
| `selected_fusion_v116_localization.png` |

**Visualizations produced by this notebook:** `fusion_validation_dashboard.png`, `selected_fusion_patient_heatmap.png`, `selected_fusion_v116_localization.png`

## Phase dataflow

```mermaid
flowchart LR
  A["inputs: mark_4d_gate_result.json, mark 1/mark_4c_outputs/"] -->
  B[phase cells: provenance + reuse/rebuild + compute]
  B --> G["gate: mark_4e_gate_result.json"]
  B --> O[organized per-phase outputs]
  G --> D[downstream notebook reads this gate]
```


## Key finding (reproduced)

**Mark 4E controlling pass.** Maximum fusion at weight **0.70** passes **all 6 final targets** — mean patient Dice, V104, V116, Q1 detection, positive predicted-empty and empty-slice FP — closing the research loop that started in Mark 1.

## Gate

`mark_4e_gate_result.json` — final gate (all 6 targets)

## Run notes

Fusion is a pure arithmetic mean of the two arms' probability maps — no training. The selected configuration is written for the one-time locked test evaluation.

> **Shared setup:** the next cell is the *identical* global-setup cell embedded in every notebook
> (paths, seeds, provenance hashes, test lock, shared helpers). REUSE mode reads frozen artifacts from
> `mark 1/`, so each notebook is deterministic and reproducible; set the `REUSE_*` / `RUN_*` flags to
> rebuild caches or retrain (GPU hours).
>
> **Ordering matters:** this phase reads the previous phase's gate JSON from the shared output folder
> (`mark_1_to_4e_outputs/…`), so run the suite in order **00 → 09**. A phase can be re-run standalone
> once its upstream gates exist (re-running the preceding notebooks regenerates them).

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
# Centralized shared output root under Evaluation/output (one folder per notebook)
SHARED_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "output"
# Legacy outputs (read-only fallback for the availability-check import helpers)
LEGACY_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
PHASE_DIR = {
    "00_setup": "00_pipeline_overview",
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
OUT       = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "data"    for phase in PHASE_DIR}
OUT_FIGS  = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "figures" for phase in PHASE_DIR}
OUT_CACHE = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "caches"  for phase in PHASE_DIR}
CONSOLIDATED = OUT["consolidated"]
CONSOLIDATED_FIGS = OUT_FIGS["consolidated"]
for _d in [*OUT.values(), *OUT_FIGS.values(), *OUT_CACHE.values()]:
    _d.mkdir(parents=True, exist_ok=True)
NOTEBOOK_KEY = "mark_4e"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Shared output helpers: availability check, cross-notebook import, registry
# ---------------------------------------------------------------------------
import shutil as _shutil

ARTIFACT_INDEX = SHARED_OUTPUT_ROOT / "artifact_index.json"


def _load_artifact_index():
    if ARTIFACT_INDEX.is_file():
        return json.loads(ARTIFACT_INDEX.read_text())
    return {"version": 1, "artifacts": []}


def _save_artifact_index(index):
    ARTIFACT_INDEX.write_text(json.dumps(index, indent=2))


def register_artifact(name, kind="data", phase=None):
    phase = phase or NOTEBOOK_KEY
    index = _load_artifact_index()
    index["artifacts"] = [a for a in index["artifacts"]
                          if not (a.get("phase") == phase and a.get("name") == name)]
    path = OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name
    index["artifacts"].append({
        "phase": phase, "name": name, "kind": kind,
        "sha256": sha256_file(path) if path.is_file() else None,
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    })
    _save_artifact_index(index)


def shared_path(phase, name, kind="data"):
    return OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name


def legacy_path(phase, name, kind="data"):
    return LEGACY_OUTPUT_ROOT / f"{phase}_outputs" / name


def load_shared(phase, name, kind="data", required=True):
    """Availability check: Evaluation/output -> legacy mark_1_to_4e_outputs -> compute/raise."""
    target = shared_path(phase, name, kind)
    if target.is_file():
        return target
    legacy = legacy_path(phase, name, kind)
    if legacy.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy2(legacy, target)
        print(f"IMPORT: reused legacy {phase}/{name} (copied to {target}).")
        return target
    if required:
        raise FileNotFoundError(
            f"Required upstream output missing: {PHASE_DIR.get(phase, phase)}/{name}.\n"
            f"Run the notebook for phase '{phase}' first (outputs land under "
            f"{SHARED_OUTPUT_ROOT / PHASE_DIR.get(phase, phase)}).")
    return None


def require_upstream_gate(phase, gate_name=None):
    gate_name = gate_name or f"{phase}_gate_result.json"
    return json.loads(load_shared(phase, gate_name, "data", required=True).read_text())


def save_figure(fig, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT_FIGS[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=170, bbox_inches="tight")
    register_artifact(name, "figures", phase)
    return target


def save_table(frame, name, phase=None, index=False):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=index)
    register_artifact(name, "data", phase)
    return target


def save_json(obj, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(obj, indent=2))
    register_artifact(name, "data", phase)
    return target


# ---------------------------------------------------------------------------
# Standardized per-phase summary dashboard
# ---------------------------------------------------------------------------
CORE_METRICS = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct",
                "empty_slice_false_positive_pct"]
TARGETS_SHEET = {phase: CONTINUATION_TARGETS for phase in
                 ["mark_1", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
GATE_SELECTOR = {
    "mark_1": "best_observed_configuration_for_diagnosis",
    "mark_2": "selected_roi_configuration",
    "mark_3": "selected_configuration",
    "mark_4": "best_metrics", "mark_4b": "selected_metrics",
    "mark_4c": "arms", "mark_4d": "selected_metrics", "mark_4e": "selected_metrics",
}
TREND_CSV = {
    "mark_1":  ("calibration_configuration_results.csv", "tumor_threshold"),
    "mark_2":  ("roi_configuration_results.csv", "liver_threshold"),
    "mark_3":  ("overfit_history.csv", "epoch"),
    "mark_4":  ("mark_4_history.csv", "epoch"),
    "mark_4b": ("threshold_results.csv", "threshold"),
    "mark_4c": ("mark_4c_history.csv", "epoch"),
    "mark_4d": ("reconciled_threshold_results.csv", "threshold"),
    "mark_4e": ("fusion_threshold_results.csv", "threshold"),
}


def _metric_color(metric, value, targets):
    if not targets or metric not in targets:
        return "#4C72B0"
    target = targets[metric]
    passed = (value <= target if metric in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else value >= target)
    return "#2E9E5B" if passed else "#C44E52"


def render_summary_dashboard(phase):
    gate = json.loads((OUT[phase] / f"{phase}_gate_result.json").read_text())
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes[0, 0].axis("off")
    text_lines = [f"phase: {phase}", f"status: {gate.get('status')}",
                  "decision: %s" % (gate.get("decision") or gate.get("next_notebook")
                                    or gate.get("next_step") or "-")]
    for key in ("test_images_accessed", "manifest_sha256", "next_mark", "next_step"):
        if key in gate:
            text_lines.append(f"{key}: {gate[key]}")
    axes[0, 0].text(0.02, 0.99, "\n".join(text_lines), transform=axes[0, 0].transAxes,
                    va="top", ha="left", fontsize=9, family="monospace")
    axes[0, 0].set_title("Gate metadata", fontsize=11, weight="bold")

    selector = GATE_SELECTOR.get(phase)
    selected = gate.get(selector) if selector else None
    targets = TARGETS_SHEET.get(phase)
    row = None
    if isinstance(selected, list):
        chosen = gate.get("selected_arm") or (selected[0].get("arm") if selected else None)
        for arm in selected:
            if arm.get("arm") == chosen:
                row = arm
    else:
        row = selected
    axes[1, 0].set_title("Selected metrics vs targets (green=pass, red=miss)",
                         fontsize=10, weight="bold")
    if row is not None:
        metric_names = [m for m in CORE_METRICS if m in row]
        if metric_names:
            values = [float(row[m]) for m in metric_names]
            axes[1, 0].bar(np.arange(len(metric_names)), values,
                           color=[_metric_color(m, float(row[m]), targets) for m in metric_names])
            axes[1, 0].axhline(0, color="k", lw=0.8)
            for metric in metric_names:
                if targets and metric in targets:
                    axes[1, 0].axhline(targets[metric], color="gray", lw=0.8, ls="--")
            axes[1, 0].set_xticks(np.arange(len(metric_names)))
            axes[1, 0].set_xticklabels(metric_names, rotation=30, ha="right", fontsize=8)
            axes[1, 0].set_ylabel("value")
            if isinstance(selected, list) and row.get("arm"):
                axes[1, 0].set_title(f"Selected arm: {row['arm']} vs targets",
                                     fontsize=10, weight="bold")
        else:
            axes[1, 0].axis("off")
            axes[1, 0].text(0.5, 0.5, "No core-metric table in gate selector",
                            ha="center", va="center")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.5, 0.5, "No selector in gate JSON", ha="center", va="center")

    csv_name, x_col = TREND_CSV.get(phase, (None, None))
    trend_path = (OUT[phase] / csv_name) if csv_name else None
    if trend_path is not None and trend_path.is_file():
        trend = pd.read_csv(trend_path)
        axes[1, 1].set_title(f"Trend: {csv_name} (x={x_col})", fontsize=10, weight="bold")
        if phase in ("mark_3", "mark_4c"):
            group_col = "configuration" if phase == "mark_3" else "arm"
            y_col = "hard_micro_dice" if phase == "mark_3" else "mean_patient_dice"
            for label, group in trend.groupby(group_col):
                axes[1, 1].plot(group[x_col], group[y_col], marker="o", ms=3, label=str(label))
            axes[1, 1].legend(fontsize=7)
        else:
            y_col = "mean_patient_dice" if "mean_patient_dice" in trend.columns else trend.columns[1]
            axes[1, 1].plot(trend[x_col], trend[y_col], marker="o", ms=3, color="#4C72B0")
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel("metric")
    else:
        axes[1, 1].axis("off")
        axes[1, 1].text(0.5, 0.5, "Trend CSV not available yet - compute the phase first",
                        ha="center", va="center")

    axes[0, 1].axis("off")
    produced = sorted(p.name for p in OUT[phase].iterdir() if p.is_file())
    inventory = "\n".join(f"- {name}" for name in produced[:20])
    axes[0, 1].text(0.02, 0.99, inventory or "(no data artifacts yet)",
                    transform=axes[0, 1].transAxes, va="top", ha="left", fontsize=8,
                    family="monospace")
    axes[0, 1].set_title(f"Produced artifacts (Evaluation/output/{PHASE_DIR[phase]}/data)",
                         fontsize=10, weight="bold")

    figure.suptitle(f"{phase} - phase summary dashboard", fontsize=15, weight="bold")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    save_figure(figure, f"{phase}_summary_dashboard.png", phase=phase)
    plt.show()
    print(f"PASS: {phase}_summary_dashboard.png -> {OUT_FIGS[phase]}")

# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {SHARED_OUTPUT_ROOT}")

Device: cuda | REUSE_CACHES=True | REUSE_HISTORY=True
PASS: provenance, split geometry, and test lock verified.
Outputs: D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output


# Part 8 — Mark 4E: Checkpoint-Fusion Validation Gate

**Original:** `mark 1/mark_4e_checkpoint_fusion_validation.ipynb`

## Question

Mark 4D found complementary behaviour: control retains V116 response; recall-loss improves positive-slice
recall, V104 and Q1. Can a **fixed probability-level fusion** satisfy all six temporary targets without
retraining?

## Key finding (reproduced) — **CONTROLLING POLICY**

- **Pixelwise maximum fusion @ threshold 0.70 PASSED ALL 6 TARGETS**:
  mean positive-patient Dice **0.3771**, V104 **0.1166**, V116 **0.0105**, Q1 **50.57%**,
  positive predicted-empty **27.45%**, empty-slice FP **5.55%**.
- Mean / 75-25 weighted fusions also passed; maximum was selected by the predeclared
  highest-mean-Dice rule → **decision `FREEZE_FUSION_POLICY_AND_THRESHOLD`**.

## Contract

- Seven policies: control, recall_loss, maximum, mean, control75_recall25, control25_recall75,
  geometric_mean — all evaluated over the same 14-threshold grid and the same nine positive patients.
- Passing authorizes only a bounded validation continuation/freeze check — **not** test access.

### 8.1 Verify Mark 4D provenance and cache alignment

In [2]:
mark4d_gate = require_upstream_gate("mark_4d")
assert mark4d_gate["test_images_accessed"] is False

CACHE_DIR = OUT_CACHE["mark_4d"]
cache_paths = {name: {int(p.stem.split("_")[-1]): p for p in (CACHE_DIR / name).glob("volume_*.npz")}
               for name in ["control", "recall_loss"]}
assert set(cache_paths["control"]) == set(cache_paths["recall_loss"])
assert len(cache_paths["control"]) == 13
for vid in cache_paths["control"]:
    with np.load(cache_paths["control"][vid], allow_pickle=False) as c, \
         np.load(cache_paths["recall_loss"][vid], allow_pickle=False) as r:
        assert c.files == r.files and c["probability"].shape == r["probability"].shape
        assert np.array_equal(c["truth"], r["truth"])
        assert np.array_equal(c["slice_index"], r["slice_index"])

positive_volumes = sorted(validation_manifest.loc[
    validation_manifest["tumor_pixels"].gt(0), "volume_id"].unique().tolist())
q1_limit = validation_manifest.loc[
    validation_manifest["tumor_pixels"].gt(0), "tumor_pixels"].quantile(.25)
assert len(positive_volumes) == 9
print(f"PASS: cache pairs aligned for 13 volumes; {len(positive_volumes)} positive patients; test locked.")

PASS: cache pairs aligned for 13 volumes; 9 positive patients; test locked.


### 8.2 Define fusion policies and evaluate over the complete population

In [3]:
POLICIES = ["control", "recall_loss", "maximum", "mean",
            "control75_recall25", "control25_recall75", "geometric_mean"]


def fuse(control, recall, policy):
    if policy == "control":
        return control
    if policy == "recall_loss":
        return recall
    if policy == "maximum":
        return np.maximum(control, recall)
    if policy == "mean":
        return .5 * control + .5 * recall
    if policy == "control75_recall25":
        return .75 * control + .25 * recall
    if policy == "control25_recall75":
        return .25 * control + .75 * recall
    if policy == "geometric_mean":
        return np.sqrt(np.clip(control, 0, 1) * np.clip(recall, 0, 1))
    raise KeyError(policy)


acc = {(policy, float(t)): {"patients": {}, "positive_empty": [], "empty_fp": [], "q1": []}
       for policy in POLICIES for t in THRESHOLDS}
slice_rows = []
for vid in sorted(cache_paths["control"]):
    with np.load(cache_paths["control"][vid], allow_pickle=False) as c, \
         np.load(cache_paths["recall_loss"][vid], allow_pickle=False) as r:
        control = c["probability"].astype(np.float32)
        recall = r["probability"].astype(np.float32)
        truth = c["truth"].astype(bool)
        truth_pixels = truth.sum(axis=(1, 2))
        positive = truth_pixels > 0
        empty = ~positive
        q1 = positive & (truth_pixels <= q1_limit)
        for policy in POLICIES:
            probability = fuse(control, recall, policy)
            for threshold in THRESHOLDS:
                pred = probability >= threshold
                key = (policy, float(threshold))
                a = acc[key]
                a["patients"][vid] = (2 * (pred & truth).sum() + 1e-6) / (pred.sum() + truth.sum() + 1e-6)
                pred_pixels = pred.sum(axis=(1, 2))
                detected = (pred & truth).any(axis=(1, 2))
                a["positive_empty"].extend(pred_pixels[positive] == 0)
                a["empty_fp"].extend(pred_pixels[empty] > 0)
                a["q1"].extend(detected[q1])
            if policy in ["control", "recall_loss", "maximum", "control75_recall25"]:
                pred = probability >= .5
                detected = (pred & truth).any(axis=(1, 2))
                for i in np.where(positive)[0]:
                    slice_rows.append({"policy": policy, "volume_id": vid,
                                       "slice_index": int(c["slice_index"][i]),
                                       "sample_id": str(c["sample_id"][i]),
                                       "truth_pixels": int(truth_pixels[i]),
                                       "detected": bool(detected[i]),
                                       "max_truth_probability": float(probability[i][truth[i]].max())})

rows, patient_rows = [], []
for (policy, threshold), a in acc.items():
    row = {"policy": policy, "threshold": threshold,
           "mean_patient_dice": float(np.mean([a["patients"][v] for v in positive_volumes])),
           "volume_104_dice": a["patients"][104], "volume_116_dice": a["patients"][116],
           "q1_detected_pct": 100 * np.mean(a["q1"]),
           "positive_predicted_empty_pct": 100 * np.mean(a["positive_empty"]),
           "empty_slice_false_positive_pct": 100 * np.mean(a["empty_fp"])}
    status = target_passes(row, CONTINUATION_TARGETS)
    row["targets_passed"] = sum(status.values())
    row["all_targets_passed"] = all(status.values())
    rows.append(row)
    for vid, dice in a["patients"].items():
        patient_rows.append({"policy": policy, "threshold": threshold, "volume_id": vid,
                             "has_tumor": vid in positive_volumes, "dice": dice})

m4e_results = pd.DataFrame(rows)
m4e_patients = pd.DataFrame(patient_rows)
m4e_slices = pd.DataFrame(slice_rows)
m4e_results.to_csv(OUT["mark_4e"] / "fusion_threshold_results.csv", index=False)
m4e_patients.to_csv(OUT["mark_4e"] / "fusion_patient_metrics.csv", index=False)
m4e_slices.to_csv(OUT["mark_4e"] / "fusion_positive_slice_diagnostic.csv", index=False)
best_by_policy = m4e_results.sort_values(
    ["all_targets_passed", "targets_passed", "mean_patient_dice",
     "empty_slice_false_positive_pct"],
    ascending=[False, False, False, True]).groupby("policy", as_index=False).first().sort_values(
    ["all_targets_passed", "targets_passed", "mean_patient_dice"], ascending=False)
best_by_policy.to_csv(OUT["mark_4e"] / "best_configuration_by_policy.csv", index=False)
display(best_by_policy)

,policy,threshold,mean_patient_dice,volume_104_dice,volume_116_dice,q1_detected_pct,positive_predicted_empty_pct,empty_slice_false_positive_pct,targets_passed,all_targets_passed
4,maximum,0.70,0.377087,0.116627,1.047355e-02,50.570342,27.447217,5.548066,6,True
5,mean,0.35,0.374802,0.117639,1.047181e-02,50.570342,27.447217,5.548066,6,True
2,control75_recall25,0.20,0.367569,0.119050,1.237261e-02,50.570342,27.351248,5.537696,6,True
1,control25_recall75,0.20,0.366512,0.130283,1.016657e-02,50.570342,26.583493,5.724360,6,True
6,recall_loss,0.60,0.376588,0.100240,7.116977e-04,47.908745,30.518234,4.791040,5,False
0,control,0.65,0.365442,0.063130,1.020738e-02,42.585551,37.044146,3.391061,5,False
3,geometric_mean,0.25,0.370786,0.050336,6.528438e-12,40.304183,40.978887,2.519963,4,False


### 8.3 Select the complete-gate winner

In [4]:
eligible = m4e_results.loc[m4e_results["all_targets_passed"]]
if eligible.empty:
    selected = m4e_results.sort_values(
        ["targets_passed", "mean_patient_dice", "empty_slice_false_positive_pct"],
        ascending=[False, False, True]).iloc[0]
    full_pass = False
else:
    selected = eligible.sort_values(
        ["mean_patient_dice", "empty_slice_false_positive_pct"],
        ascending=[False, True]).iloc[0]
    full_pass = True
selected_passes = target_passes(selected, CONTINUATION_TARGETS)
selection = pd.DataFrame([{"metric": key, "actual": selected[key], "target": target,
                           "direction": ("<=" if key in ["positive_predicted_empty_pct",
                                                         "empty_slice_false_positive_pct"] else ">="),
                           "passed": selected_passes[key]}
                          for key, target in CONTINUATION_TARGETS.items()])
selection.to_csv(OUT["mark_4e"] / "selected_gate_table.csv", index=False)
print("Selected:", selected.policy, "threshold", selected.threshold, "| full pass:", full_pass)
display(selection)

Selected: maximum threshold 0.699999988079071 | full pass: True


,metric,actual,target,direction,passed
0,mean_patient_dice,0.377087,0.3329,>=,True
1,volume_104_dice,0.116627,0.0500,>=,True
2,volume_116_dice,0.010474,0.0100,>=,True
3,q1_detected_pct,50.570342,35.0000,>=,True
4,positive_predicted_empty_pct,27.447217,35.0000,<=,True
5,empty_slice_false_positive_pct,5.548066,20.0000,<=,True


### 8.4 Fusion frontiers + selected patient heatmap

In [5]:
figure, axes = plt.subplots(2, 3, figsize=(20, 11))
fields = [("mean_patient_dice", "Positive-patient mean Dice", .3329),
          ("volume_116_dice", "V116 Dice", .01),
          ("q1_detected_pct", "Q1 detection (%)", 35),
          ("positive_predicted_empty_pct", "Positive predicted-empty (%)", 35),
          ("empty_slice_false_positive_pct", "Empty-slice FP (%)", 20)]
for ax, (field, title, target) in zip(axes.ravel()[:5], fields):
    for policy, group in m4e_results.groupby("policy"):
        ax.plot(group.threshold, group[field], marker="o", ms=3, label=policy)
    ax.axhline(target, ls="--", c="black")
    ax.set_title(title)
    ax.set_xlabel("Threshold")
axes[0, 0].legend(fontsize=8, ncol=2)
axes[1, 2].scatter(m4e_results.empty_slice_false_positive_pct,
                   m4e_results.positive_predicted_empty_pct,
                   c=m4e_results.mean_patient_dice, cmap="viridis", s=28)
axes[1, 2].axhline(35, ls="--", c="black"); axes[1, 2].axvline(20, ls="--", c="black")
axes[1, 2].set_xlabel("Empty FP (%)"); axes[1, 2].set_ylabel("Positive empty (%)")
axes[1, 2].set_title("Recall–specificity frontier")
figure.suptitle("Mark 4E checkpoint-fusion validation")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4e"] / "fusion_validation_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

selected_patients = m4e_patients.loc[
    (m4e_patients["policy"] == selected.policy)
    & np.isclose(m4e_patients["threshold"], selected.threshold)
    & m4e_patients["has_tumor"]]
control_patients = m4e_patients.loc[
    (m4e_patients["policy"] == "control") & np.isclose(m4e_patients["threshold"], .5)
    & m4e_patients["has_tumor"]]
recall_patients = m4e_patients.loc[
    (m4e_patients["policy"] == "recall_loss") & np.isclose(m4e_patients["threshold"], .5)
    & m4e_patients["has_tumor"]]
plot_data = pd.concat([
    control_patients.assign(configuration="control t=.50"),
    recall_patients.assign(configuration="recall t=.50"),
    selected_patients.assign(configuration=f"{selected.policy} t={selected.threshold:.2f}")])
pivot = plot_data.pivot(index="volume_id", columns="configuration", values="dice")
figure, axes = plt.subplots(figsize=(9, 7))
image = axes.imshow(pivot.values, aspect="auto", cmap="viridis",
                    vmin=0, vmax=max(.7, float(pivot.max().max())))
axes.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=20)
axes.set_yticks(range(len(pivot.index)), pivot.index)
axes.set_title("Positive-patient Dice: baselines vs selected fusion")
figure.colorbar(image, ax=axes, label="Dice")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4e"] / "selected_fusion_patient_heatmap.png",
               dpi=170, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_3324\1130045033.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\alanm\AppData\Local\Temp\ipykernel_3324\1130045033.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.5 Selected-fusion V116 localization + write the Mark 4E gate

In [6]:
with np.load(cache_paths["control"][116], allow_pickle=False) as c, \
     np.load(cache_paths["recall_loss"][116], allow_pickle=False) as r:
    probability = fuse(c["probability"].astype(np.float32), r["probability"].astype(np.float32),
                       selected.policy)
    truth = c["truth"].astype(bool)
    pred = probability >= selected.threshold
    truth_pixels = truth.sum(axis=(1, 2))
    detected = (pred & truth).any(axis=(1, 2))
    candidates = np.where((truth_pixels > 0) & (~detected))[0]
    focus = candidates[np.argsort(truth_pixels[candidates])[-4:]][::-1]
    lookup = validation_manifest.set_index("sample_id")
    figure, axes = plt.subplots(len(focus), 5, figsize=(18, 4 * len(focus)), squeeze=False)
    for row_axes, i in zip(axes, focus):
        sid = str(c["sample_id"][i])
        row = lookup.loc[sid]
        with Image.open(DATASET_ROOT / row.image_path) as handle:
            image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
        panels = [(image, "CT", "gray"), (truth[i], "Truth", "gray"),
                  (c["probability"][i], "Control probability", "magma"),
                  (r["probability"][i], "Recall probability", "magma"),
                  (probability[i], f"{selected.policy} probability", "magma")]
        for ax, (panel, title, cmap) in zip(row_axes, panels):
            ax.imshow(panel, cmap=cmap, vmin=0, vmax=1)
            ax.set_title(f"{title} | slice {int(c['slice_index'][i])}")
            ax.axis("off")
    figure.suptitle("V116 selected-fusion missed slices")
    figure.tight_layout()
    figure.savefig(OUT_FIGS["mark_4e"] / "selected_fusion_v116_localization.png",
                   dpi=170, bbox_inches="tight")
    plt.show()

if full_pass:
    decision, next_notebook = "FREEZE_FUSION_POLICY_AND_THRESHOLD", "mark_4f_fusion_freeze_and_bounded_confirmation"
else:
    decision, next_notebook = "NO_FUSION_PASS_RUN_MODERATE_ALPHA_AND_HARD_POSITIVE_ABLATION", "mark_4f_targeted_training_ablation"

m4e_gate = {
    "status": "mark_4e_fusion_pass" if full_pass else "mark_4e_fusion_fail",
    "selected_policy": str(selected.policy),
    "selected_threshold": float(selected.threshold),
    "selected_metrics": {key: float(selected[key]) for key in CONTINUATION_TARGETS},
    "targets_passed": int(selected.targets_passed),
    "decision": decision, "next_notebook": next_notebook,
    "metric_definition": "mean Dice over nine tumour-positive validation patients",
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "test_images_accessed": False,
}
(OUT["mark_4e"] / "mark_4e_gate_result.json").write_text(json.dumps(m4e_gate, indent=2))
display(pd.DataFrame([m4e_gate]).T.rename(columns={0: "value"}))
print(json.dumps(m4e_gate, indent=2))

# ---- Reproduction check against the original gate ----
orig_m4e = json.loads((MARK1_DIR / "mark_4e_outputs" / "mark_4e_gate_result.json").read_text())
diffs = {k: abs(float(m4e_gate["selected_metrics"][k]) - float(orig_m4e["selected_metrics"][k]))
         for k in CONTINUATION_TARGETS}
print("Mark 4E reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 4E gate drifted from the original!"
assert m4e_gate["selected_policy"] == orig_m4e["selected_policy"]
assert abs(float(m4e_gate["selected_threshold"]) - float(orig_m4e["selected_threshold"])) < 1e-5
assert m4e_gate["status"] == orig_m4e["status"]
print("PASS: Mark 4E gate matches the original mark_4e_gate_result.json.")

C:\Users\alanm\AppData\Local\Temp\ipykernel_3324\4152072498.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,value
status,mark_4e_fusion_pass
selected_policy,maximum
selected_threshold,0.7
selected_metrics,"{'mean_patient_dice': 0.3770866927494516, 'vol..."
targets_passed,6
decision,FREEZE_FUSION_POLICY_AND_THRESHOLD
next_notebook,mark_4f_fusion_freeze_and_bounded_confirmation
metric_definition,mean Dice over nine tumour-positive validation...
manifest_sha256,575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63...
test_images_accessed,False


{
  "status": "mark_4e_fusion_pass",
  "selected_policy": "maximum",
  "selected_threshold": 0.699999988079071,
  "selected_metrics": {
    "mean_patient_dice": 0.3770866927494516,
    "volume_104_dice": 0.11662697526259982,
    "volume_116_dice": 0.010473553322754188,
    "q1_detected_pct": 50.57034220532319,
    "positive_predicted_empty_pct": 27.447216890595012,
    "empty_slice_false_positive_pct": 5.548065954578451
  },
  "targets_passed": 6,
  "decision": "FREEZE_FUSION_POLICY_AND_THRESHOLD",
  "next_notebook": "mark_4f_fusion_freeze_and_bounded_confirmation",
  "metric_definition": "mean Dice over nine tumour-positive validation patients",
  "manifest_sha256": "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889",
  "test_images_accessed": false
}
Mark 4E reproduction check: {'mean_patient_dice': 0.0, 'volume_104_dice': 0.0, 'volume_116_dice': 0.0, 'q1_detected_pct': 0.0, 'positive_predicted_empty_pct': 0.0, 'empty_slice_false_positive_pct': 0.0}
PASS: Mark 4E gate 

In [7]:

# ---- Standard phase summary dashboard (centralized visualization) ----
render_summary_dashboard("mark_4e")


PASS: mark_4e_summary_dashboard.png -> D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\08_mark_4e\figures


C:\Users\alanm\AppData\Local\Temp\ipykernel_3324\4248019987.py:384: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ---------------------------------------------------------------------------
# Publish key mark_4e artifacts to the shared/legacy folder other code reads.
#
# External consumers: `mark 1 (part 2)/AGENTS.md`, `05_ARTIFACT_INDEX_AND_REPRODUCIBILITY.md` (`mark_4e_gate_result.json`, `fusion_*`).
# These code files hardcode artifact paths under `mark 1/mark_4e_outputs/`, so
# after every run the freshly produced artifacts are mirrored there to keep
# those code files working. Values are recomputed from frozen inputs and
# verified against the original gates (reproduction check above), so the
# mirrored files are equivalent.
# ---------------------------------------------------------------------------
import shutil

PUBLISH_DIR = MARK1_DIR / "mark_4e_outputs"
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

published = []
for pattern in ("*.csv", "*.json", "*.pth"):
    for source in sorted(OUT["mark_4e"].glob(pattern)):
        shutil.copy2(source, PUBLISH_DIR / source.name)
        published.append(PUBLISH_DIR / source.name)

assert published, f"no mark_4e artifacts found to publish"
print(f"PUBLISHED {len(published)} mark_4e artifacts to {PUBLISH_DIR}:")
for artifact in sorted(published):
    print("  " + str(artifact))

PUBLISHED 6 mark_4e artifacts to D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4e_outputs:
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4e_outputs\best_configuration_by_policy.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4e_outputs\fusion_patient_metrics.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4e_outputs\fusion_positive_slice_diagnostic.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4e_outputs\fusion_threshold_results.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4e_outputs\mark_4e_gate_result.json
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4e_outputs\selected_gate_table.csv
